In [13]:
import sys
import os
from pathlib import Path

import pandas as pd
import duckdb
import pyarrow.parquet as pq

sys.path.append(os.path.abspath('..'))

from utilities import customdataloader

In [19]:
DATA_ROOT = Path(r"G:\My Drive\Data")


# Ajouter les nouveaux fichiers de 2024 aux anciens pour avoir des fichiers complets
prices_hist = pd.read_parquet(DATA_ROOT / "prices" / "prices.parquet")
prices_2024 = pd.read_parquet(DATA_ROOT / "prices" / "prices_2024.parquet")
prices_all = pd.concat([prices_hist, prices_2024], ignore_index=True)

costs_hist = pd.read_parquet(DATA_ROOT / "costs" / "costs.parquet")
costs_2024 = pd.read_parquet(DATA_ROOT / "costs" / "costs_2024.parquet")
costs_all = pd.concat([costs_hist, costs_2024], ignore_index=True)

prices_all.to_parquet(DATA_ROOT / "prices" / "prices_all.parquet", index=False)
costs_all.to_parquet(DATA_ROOT / "costs" / "costs_all.parquet", index=False)

loader = customdataloader(
    pricefile=str(DATA_ROOT / "prices" / "prices_all.parquet"),
    costfile=str(DATA_ROOT / "costs" / "costs_all.parquet"),
    dsimfile=[
        str(DATA_ROOT / "sim_daily" / f"sim_daily_{y}.parquet")
        for y in [2020, 2021, 2022, 2023, 2024]
    ],
    msimfile=[
        str(DATA_ROOT / "sim_monthly" / f"sim_monthly_{y}.parquet")
        for y in [2020, 2021, 2022, 2023, 2024]
    ],
)


print(f"costs:  {costs_all.shape[0]:,} rows")
print(f"prices: {prices_all.shape[0]:,} rows")

costs:  11,844 rows
prices: 597,422 rows


In [20]:
#test du loader 
TARGET_MONTH = "2020-02"
CUTOFF_DATE = "2020-01-07"

# Construire les candidats a partir de la simulation mensuelle du mois cible 
sim_file = DATA_ROOT / "sim_monthly" / f"sim_monthly_{TARGET_MONTH[:4]}.parquet"

sim_eids = duckdb.query(f"""
    SELECT DISTINCT EID
    FROM read_parquet('{sim_file}')
    WHERE strftime(DATETIME, '%Y-%m') = '{TARGET_MONTH}'
""").to_df()

known_eids = sim_eids["EID"].values

filterdf = pd.DataFrame({
    "EID": list(known_eids) * 2,   
    "MONTH": [TARGET_MONTH] * len(known_eids) * 2,
    "PEAKID": [0] * len(known_eids) + [1] * len(known_eids),
})

print(f"Candidats pour {TARGET_MONTH}: {len(filterdf)}")
filterdf.head()

Candidats pour 2020-02: 9076


,EID,MONTH,PEAKID
0,635,2020-02,0
1,637,2020-02,0
2,706,2020-02,0
3,720,2020-02,0
4,732,2020-02,0


In [22]:
# Charger les simulations mensuelles pour les candidats
monthly_sim = loader.get_monthly_data(filterdf)
print("monthly_sim shape:", monthly_sim.shape)
monthly_sim.head()

monthly_sim shape: (9475344, 13)


,SCENARIOID,EID,DATETIME,PEAKID,m_ACTIVATIONLEVEL,m_WINDIMPACT,m_SOLARIMPACT,m_HYDROIMPACT,m_NONRENEWBALIMPACT,m_EXTERNALIMPACT,m_LOADIMPACT,m_TRANSMISSIONOUTAGEIMPACT,m_PSM
0,1,501,2020-02-03 01:00:00,0,3.2879,1.7152,0.0,1.0667,3.8504,-3.3935,2.4749,0.6233,0.0
1,1,501,2020-02-03 02:00:00,0,4.3435,2.4034,0.0,1.0636,3.8575,-3.4181,2.8692,0.5360,0.0
2,1,501,2020-02-03 03:00:00,0,5.6095,3.3187,0.0,1.0629,3.8893,-3.3806,3.1396,0.5304,0.0
3,1,501,2020-02-03 04:00:00,0,6.6908,4.2450,0.0,1.0650,3.9553,-3.3368,3.2513,0.5317,0.0
4,1,501,2020-02-03 05:00:00,0,7.3176,4.4017,0.0,1.0706,4.3250,-3.2563,3.4031,0.5030,0.0


In [23]:
# Charger les simulations journalières pour les candidats
daily_filterdf = pd.DataFrame({
    "EID": list(known_eids) * 2,
    "MONTH": [CUTOFF_DATE[:7]] * len(known_eids) * 2,
    "PEAKID": [0] * len(known_eids) + [1] * len(known_eids),
})

daily_sim = loader.get_daily_data(daily_filterdf, cutoff_date=CUTOFF_DATE)
print("daily_sim shape:", daily_sim.shape)
daily_sim.head()

daily_sim shape: (2287152, 13)


,SCENARIOID,EID,DATETIME,PEAKID,d_ACTIVATIONLEVEL,d_WINDIMPACT,d_SOLARIMPACT,d_HYDROIMPACT,d_NONRENEWBALIMPACT,d_EXTERNALIMPACT,d_LOADIMPACT,d_TRANSMISSIONOUTAGEIMPACT,d_PSD
0,1,634,2020-01-01 01:00:00,0,14.3247,-3.2806,0.0,-1.1631,2.3503,-1.1220,-0.2012,-0.5600,0.0
1,1,634,2020-01-01 02:00:00,0,13.2333,-3.3271,0.0,-1.1446,1.8813,-1.2757,-0.1414,-0.5698,0.0
2,1,634,2020-01-01 03:00:00,0,12.7973,-3.2476,0.0,-1.1368,1.7381,-1.4462,-0.1083,-0.5602,0.0
3,1,634,2020-01-01 04:00:00,0,12.7341,-3.1504,0.0,-1.1338,1.7924,-1.4205,-0.2835,-0.5595,0.0
4,1,634,2020-01-01 05:00:00,0,12.8675,-3.0634,0.0,-1.1467,2.0082,-1.6333,-0.3352,-0.5523,0.0


In [24]:
print(daily_sim["DATETIME"].max())   # Verif importante pour voir si cutoff date est bien appliquée 

2020-01-08 00:00:00
